# V19.0-pivot — پیاده‌سازی کامل فاز ۰: PINN دوشاخه با پارامتری‌سازی ناحیه‌ای Cf

**وضعیت:** پیاده‌سازی تازه، مطابق مشخصات گزارش V19 و HANDOFF_MEMO — نه کپی کد قبلی شما.
قبل از اجرای `medium`، حتماً یک بار با `RUN_MODE="smoke"` اجرا کنید تا از سلامت کد مطمئن شوید.

هر سلول کد، دقیقاً معادل یکی از Cell های ۱ تا ۹ توصیف‌شده در HANDOFF_MEMO است (در کامنت بالای هر سلول مشخص شده).

## Cell 1 — راه‌اندازی، وارد کردن کتابخانه‌ها، RUN_MODE

In [1]:
# ============================================================
# Cell 1 — راه‌اندازی
# ============================================================
import time
import pickle
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)  # دقت مضاعف -- برای پایداری عددی حل‌گر حیاتی است

PATCH_LEVEL = "V19.0-pivot / phase0"  # مهر provenance -- در هر خروجی ذخیره می‌شود

# سه سطح اجرا -- هرگز مستقیم medium/full را بدون یک بار smoke اجرا نکنید
RUN_MODE = "smoke"   # یکی از: "smoke", "medium", "full"

_RUN_MODE_TABLE = {
    #              nx, ny,  n_steps,  n_coll,  n_obs,  adam_ep1, adam_ep2, lbfgs_iters
    "smoke":  dict(nx=12, ny=6,  n_steps=60,  n_coll=256,  n_obs=15, adam_ep1=50,   adam_ep2=50,   lbfgs_iters=20),
    "medium": dict(nx=24, ny=12, n_steps=300, n_coll=2000, n_obs=40, adam_ep1=3000, adam_ep2=3000, lbfgs_iters=300),
    "full":   dict(nx=48, ny=24, n_steps=600, n_coll=6000, n_obs=80, adam_ep1=8000, adam_ep2=8000, lbfgs_iters=1000),
}
RC = _RUN_MODE_TABLE[RUN_MODE]
print(f"[Cell 1] RUN_MODE={RUN_MODE} -> {RC}")

SEED = 0
_rng_np = np.random.default_rng(SEED)
_key = jax.random.PRNGKey(SEED)
def next_key():
    global _key
    _key, sub = jax.random.split(_key)
    return sub


[Cell 1] RUN_MODE=smoke -> {'nx': 12, 'ny': 6, 'n_steps': 60, 'n_coll': 256, 'n_obs': 15, 'adam_ep1': 50, 'adam_ep2': 50, 'lbfgs_iters': 20}


## Cell 2 — پیکربندی (فیزیک، شبکه‌بندی، annealing، پرچم‌های سناریو)

In [2]:
# ============================================================
# Cell 2 — پیکربندی
# ============================================================
# --- شبکه‌بندی و فیزیک (همان مقیاس اسکریپت شناسایی‌پذیری قبلی) ---
NX, NY = RC["nx"], RC["ny"]
LX, LY = 150e3, 50e3
DX, DY = LX / NX, LY / NY
G = 9.81
H0 = 80.0
F0 = 6.5e-5
U0 = 0.15
OMEGA = 2 * jnp.pi / (12.42 * 3600.0)
DT = 60.0
N_STEPS = RC["n_steps"]

_c = float(jnp.sqrt(G * H0))
_cfl = DT * _c / min(DX, DY)
print(f"[Cell 2] CFL number = {_cfl:.3f} (باید کمتر از ~۱ باشد)")

# --- اصلاحات باگ V17.7.1 (بدون تغییر، طبق دفتر باگ) ---
VELOCITY_REG_CAP_MPS = 6.0     # سقف سرعت اصلاح‌شده (باگ #۱)
FRIC_SIGMOID_TEMP0 = 3.0       # دمای اولیه‌ی annealing (باگ #۲) -- k شروع می‌کند از این مقدار

# --- annealing سطح-صفر (بخش ۵.۱ گزارش، گزینه‌ی الف) ---
N_ZONES = 2                    # فاز ۰: دو ناحیه، مطابق اولویت گزارش
K_MIN = FRIC_SIGMOID_TEMP0
K_MAX = 25.0
K_ANNEAL_EPOCHS = RC["adam_ep2"]   # annealing فقط طی فاز ۲ (پس از آزادسازی فیزیک) انجام می‌شود

def k_schedule(epoch_in_phase2):
    """annealing خطی از K_MIN به K_MAX طی فاز ۲. epoch_in_phase2 از ۰ شروع می‌شود."""
    frac = jnp.clip(epoch_in_phase2 / K_ANNEAL_EPOCHS, 0.0, 1.0)
    return K_MIN + (K_MAX - K_MIN) * frac

ZONE_SEPARATION_WEIGHT = 5.0
ZONE_SEPARATION_MARGIN = 1.5e-3   # حداقل فاصله‌ی مطلوب |Cf_high - Cf_low|

CF_BACKGROUND = 2.5e-3
CF_LOW_TRUE = 1.5e-3
CF_HIGH_TRUE = 4.0e-3

# --- گذار Freeze-Release (بخش ۵.۳) ---
WARM_TRANSITION_EPOCHS = 200    # طول بازه‌ی میرایی نمایی گشتاورهای Adam شاخه‌ی اصطکاک
MOMENT_DECAY_RATE = 0.97        # ضریب میرایی هر epoch طی بازه‌ی گذار (0.97**200 ≈ 0.0024 -> عملا صفر، اما نرم)

# --- پرچم‌های سناریو (برای Cell 9 / ablation) ---
WIND_ON = False
DIFF_ON = False
OBSERVE_VELOCITY = False   # True فقط برای بازوی کنترل تشخیصی

WIND_AMPLITUDE = 3e-6      # (m/s^2) بزرگی جمله‌ی شتاب باد -- در مقیاس اصطکاک/فشار (بخش ۳.۱ گزارش: ~۹٪ از جمله‌ی غالب)
NU_DIFF = 5.0              # (m^2/s) ضریب لزجت گردابه‌ای برای جمله‌ی نفوذ (فقط وقتی DIFF_ON=True)

N_COLLOCATION = RC["n_coll"]
N_OBS_PTS = RC["n_obs"]
OBS_NOISE_STD_FRAC = 0.03   # ۳٪ نوفه‌ی مشاهداتی (بخش ۳.۱ گزارش)

ADAM_EPOCHS_PHASE1 = RC["adam_ep1"]
ADAM_EPOCHS_PHASE2 = RC["adam_ep2"]
LBFGS_MAXITER = RC["lbfgs_iters"]

PHYS_WEIGHT_PHASE1 = 0.01
PHYS_WEIGHT_PHASE2 = 1.0

print(f"[Cell 2] patch_level={PATCH_LEVEL}, N_ZONES={N_ZONES}, "
      f"VELOCITY_REG_CAP_MPS={VELOCITY_REG_CAP_MPS}, WARM_TRANSITION_EPOCHS={WARM_TRANSITION_EPOCHS}")


[Cell 2] CFL number = 0.202 (باید کمتر از ~۱ باشد)
[Cell 2] patch_level=V19.0-pivot / phase0, N_ZONES=2, VELOCITY_REG_CAP_MPS=6.0, WARM_TRANSITION_EPOCHS=200


## Cell 3 — حل‌گر مرجع و تولید داده‌ی دوقلوی مصنوعی

In [3]:
# ============================================================
# Cell 3 — حل‌گر مرجع (Arakawa C-grid) + تولید داده‌ی دوقلوی مصنوعی
# ============================================================
# همان طرح عددی اسکریپت شناسایی‌پذیری (validated)، به‌علاوه‌ی جمله‌های
# اختیاری باد و نفوذ. معادلات (بخش ۳ گزارش V19):
#   du/dt = -g deta/dx + f v - (Cf/H0) u sqrt(u^2+v^2) + wind_x + nu*Lap(u)
#   dv/dt = -g deta/dy - f u - (Cf/H0) v sqrt(u^2+v^2) + wind_y + nu*Lap(v)
#   deta/dt = -H0 (du/dx + dv/dy)

def interp_v_to_u(v):
    v_pad = jnp.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_center = 0.5 * (v_pad[:, :-1] + v_pad[:, 1:])
    return 0.5 * (v_center[:-1, :] + v_center[1:, :])

def interp_u_to_v(u):
    u_center = 0.5 * (u[:-1, :] + u[1:, :])
    u_pad = jnp.pad(u_center, ((0, 0), (1, 1)), mode="edge")
    return 0.5 * (u_pad[:, :-1] + u_pad[:, 1:])

def cf_at_u(Cf):
    idx = jnp.clip(jnp.arange(NX + 1), 0, NX - 1)
    return Cf[idx, :]

def cf_at_v(Cf):
    idx = jnp.clip(jnp.arange(NY + 1), 0, NY - 1)
    return Cf[:, idx]

# --- الگوی مکانی باد: عمداً با مرز واقعی Cf ناهم‌راستا (برای آزمون H1b) ---
_yy_u = (jnp.arange(NY) + 0.5) * DY
WIND_PATTERN_U = jnp.sin(2 * jnp.pi * _yy_u / LY)[None, :]  # (1,NY) -> broadcast روی x
_yy_v = (jnp.arange(NY + 1)) * DY
WIND_PATTERN_V = jnp.zeros((NX, NY + 1))  # فقط مؤلفه‌ی x فعال است، برای سادگی

def laplacian_u(u):
    u_pad_x = jnp.pad(u, ((1, 1), (0, 0)), mode="edge")
    u_pad_y = jnp.pad(u, ((0, 0), (1, 1)), mode="edge")
    d2x = (u_pad_x[2:, :] - 2 * u + u_pad_x[:-2, :]) / DX**2
    d2y = (u_pad_y[:, 2:] - 2 * u + u_pad_y[:, :-2]) / DY**2
    return d2x + d2y

def laplacian_v(v):
    v_pad_x = jnp.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_pad_y = jnp.pad(v, ((0, 0), (1, 1)), mode="edge")
    d2x = (v_pad_x[2:, :] - 2 * v + v_pad_x[:-2, :]) / DX**2
    d2y = (v_pad_y[:, 2:] - 2 * v + v_pad_y[:, :-2]) / DY**2
    return d2x + d2y

def shallow_water_step(state, t, Cf, wind_on, diff_on):
    eta, u, v = state
    Cf_u, Cf_v = cf_at_u(Cf), cf_at_v(Cf)
    v_at_u, u_at_v = interp_v_to_u(v), interp_u_to_v(u)
    speed_u = jnp.sqrt(u**2 + v_at_u**2 + 1e-8)
    speed_v = jnp.sqrt(u_at_v**2 + v**2 + 1e-8)

    deta_dx = (eta[1:, :] - eta[:-1, :]) / DX
    deta_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fric_u = Cf_u / H0 * u * speed_u
    fric_v = Cf_v / H0 * v * speed_v

    wind_u = jnp.where(wind_on, WIND_AMPLITUDE * WIND_PATTERN_U, 0.0)
    wind_v = jnp.where(wind_on, 0.0, 0.0)
    diff_u = jnp.where(diff_on, NU_DIFF * laplacian_u(u), 0.0)
    diff_v = jnp.where(diff_on, NU_DIFF * laplacian_v(v), 0.0)

    u_interior_new = u[1:-1, :] + DT * (-G * deta_dx + F0 * v_at_u[1:-1, :]
                                          - fric_u[1:-1, :] + wind_u[1:-1, :] + diff_u[1:-1, :])
    u_west = jnp.full((1, NY), U0 * jnp.cos(OMEGA * t))
    u_east = jnp.zeros((1, NY))
    u_new = jnp.concatenate([u_west, u_interior_new, u_east], axis=0)

    v_interior_new = v[:, 1:-1] + DT * (-G * deta_dy - F0 * u_at_v[:, 1:-1]
                                          - fric_v[:, 1:-1] + wind_v[:, 1:-1] + diff_v[:, 1:-1])
    v_south = jnp.zeros((NX, 1))
    v_north = jnp.zeros((NX, 1))
    v_new = jnp.concatenate([v_south, v_interior_new, v_north], axis=1)

    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    eta_new = eta - DT * H0 * div
    return (eta_new, u_new, v_new)

def true_cf_field():
    """میدان حقیقی مصنوعی: دو ناحیه با مرز مورب (بخش ۴.۱ گزارش)."""
    xc = (jnp.arange(NX) + 0.5) * DX
    yc = (jnp.arange(NY) + 0.5) * DY
    Xc, Yc = jnp.meshgrid(xc, yc, indexing="ij")
    boundary = 0.5 * LX + 0.3 * (Yc - 0.5 * LY)  # مرز مورب
    return jnp.where(Xc < boundary, CF_LOW_TRUE, CF_HIGH_TRUE)

def run_reference_solver(Cf, wind_on, diff_on, n_steps=None):
    n_steps = n_steps or N_STEPS
    eta = jnp.zeros((NX, NY)); u = jnp.zeros((NX + 1, NY)); v = jnp.zeros((NX, NY + 1))
    def body(state, n):
        t = n * DT
        new_state = shallow_water_step(state, t, Cf, wind_on, diff_on)
        eta, u, v = new_state
        u_c = 0.5 * (u[:-1, :] + u[1:, :])
        v_c = 0.5 * (v[:, :-1] + v[:, 1:])
        return new_state, (eta, u_c, v_c)
    _, (eta_traj, u_traj, v_traj) = lax.scan(body, (eta, u, v), jnp.arange(n_steps))
    return eta_traj, u_traj, v_traj   # هر کدام (n_steps, NX, NY)

print("[Cell 3] در حال اجرای حل‌گر مرجع برای تولید داده‌ی دوقلوی مصنوعی ...")
CF_TRUE = true_cf_field()
_t0 = time.time()
ETA_TRUE, U_TRUE, V_TRUE = run_reference_solver(CF_TRUE, WIND_ON, DIFF_ON)
print(f"[Cell 3] اجرا در {time.time()-_t0:.1f} ثانیه تمام شد. شکل خروجی: {ETA_TRUE.shape}")

# --- چاپ فوری توزیع سرعت واقعی (طبق Cell 3 اصلی در V17.7.1) ---
_speed = np.asarray(jnp.sqrt(U_TRUE**2 + V_TRUE**2))
print(f"[Cell 3] میانه‌ی سرعت واقعی = {np.median(_speed):.3f} m/s، "
      f"حداکثر = {_speed.max():.3f} m/s، "
      f"٪ نقاط بالای سقف {VELOCITY_REG_CAP_MPS} m/s = {100*np.mean(_speed > VELOCITY_REG_CAP_MPS):.2f}٪")


[Cell 3] در حال اجرای حل‌گر مرجع برای تولید داده‌ی دوقلوی مصنوعی ...


TypeError: add got incompatible shapes for broadcasting: (11, 6), (0, 6).

## Cell 4 — دیتاست (نقاط مشاهده‌ی پراکنده، نقاط کولوکیشن، assert سلامت)

In [ ]:
# ============================================================
# Cell 4 — دیتاست + assert سلامت
# ============================================================
_rng4 = np.random.default_rng(SEED + 1)

# --- نقاط پراکنده‌ی "رد ماهواره‌ای" (ثابت طی کل آموزش) ---
_obs_idx_flat = _rng4.choice(NX * NY, size=N_OBS_PTS, replace=False)
_ix_obs, _iy_obs = np.unravel_index(_obs_idx_flat, (NX, NY))
_xc = (np.arange(NX) + 0.5) * DX
_yc = (np.arange(NY) + 0.5) * DY
OBS_X = jnp.array(_xc[_ix_obs])
OBS_Y = jnp.array(_yc[_iy_obs])

_snapshot_steps = np.linspace(N_STEPS // 4, N_STEPS - 1, 4).astype(int)
OBS_T = jnp.array(_snapshot_steps * DT)

def _sample_traj(traj, ix, iy, steps):
    # traj: (n_steps, NX, NY) -> (n_steps_sel, n_pts) via paired fancy indexing
    arr = np.asarray(traj)[steps]        # (n_steps_sel, NX, NY)
    return arr[:, ix, iy]                # (n_steps_sel, n_pts)

_h_clean = _sample_traj(ETA_TRUE, _ix_obs, _iy_obs, _snapshot_steps)   # (n_snap, n_obs)
_u_clean = _sample_traj(U_TRUE, _ix_obs, _iy_obs, _snapshot_steps)
_v_clean = _sample_traj(V_TRUE, _ix_obs, _iy_obs, _snapshot_steps)

_noise_std = OBS_NOISE_STD_FRAC * np.std(_h_clean)
H_OBS = jnp.array(_h_clean + _rng4.normal(scale=_noise_std, size=_h_clean.shape))
U_OBS = jnp.array(_u_clean + _rng4.normal(scale=_noise_std, size=_u_clean.shape))
V_OBS = jnp.array(_v_clean + _rng4.normal(scale=_noise_std, size=_v_clean.shape))

print(f"[Cell 4] {N_OBS_PTS} نقطه‌ی پراکنده × {len(_snapshot_steps)} زمان‌سنجه = "
      f"{N_OBS_PTS*len(_snapshot_steps)} مشاهده‌ی h (نوفه‌ی std={_noise_std:.4g})")

# --- نقاط کولوکیشن فیزیک (تصادفی در دامنه‌ی مکان-زمان) ---
_key_c = next_key()
COLL_X = jax.random.uniform(_key_c, (N_COLLOCATION,), minval=0.0, maxval=LX)
_key_c = next_key()
COLL_Y = jax.random.uniform(_key_c, (N_COLLOCATION,), minval=0.0, maxval=LY)
_key_c = next_key()
COLL_T = jax.random.uniform(_key_c, (N_COLLOCATION,), minval=0.0, maxval=(N_STEPS - 1) * DT)

# --- assert سلامت: سازگاری سقف سرعت (طبق Cell 4 اصلی V17.7.1) ---
_max_speed_true = float(_speed.max())
assert _max_speed_true < VELOCITY_REG_CAP_MPS * 1.5, (
    f"سقف سرعت ({VELOCITY_REG_CAP_MPS} m/s) با دامنه‌ی دینامیک واقعی ({_max_speed_true:.2f} m/s) "
    "ناسازگار است -- سقف را افزایش دهید یا مولد داده را بازبینی کنید."
)
print(f"[Cell 4] assert سقف سرعت: OK ({_max_speed_true:.3f} m/s < {1.5*VELOCITY_REG_CAP_MPS:.1f} m/s)")

# نرمال‌سازی ورودی‌های شبکه (مهم برای پایداری آموزش PINN)
def normalize_xyt(x, y, t):
    return x / LX, y / LY, t / ((N_STEPS - 1) * DT)


## Cell 5 — معماری PINN دوشاخه (شاخه‌ی هیدرودینامیک + شاخه‌ی اصطکاک سطح-صفر annealed)
این تنها سلولی است که طبق گزارش V19 باید از پایه بازطراحی شود؛ فرمول دقیقاً بخش ۵.۲ است:
`Cf(x,y) = Cf_low + (Cf_high - Cf_low) * sigmoid(k * phi(x,y))`

In [ ]:
# ============================================================
# Cell 5 — معماری دوشاخه (پیاده‌سازی تازه، طبق فرمول بخش ۵.۲ گزارش)
# ============================================================
def init_mlp(key, layer_sizes):
    params = []
    keys = jax.random.split(key, len(layer_sizes) - 1)
    for k, n_in, n_out in zip(keys, layer_sizes[:-1], layer_sizes[1:]):
        wk, _ = jax.random.split(k)
        W = jax.random.normal(wk, (n_in, n_out)) * jnp.sqrt(2.0 / n_in)
        b = jnp.zeros(n_out)
        params.append((W, b))
    return params

def mlp_apply(params, x):
    a = x
    for i, (W, b) in enumerate(params):
        z = a @ W + b
        a = jnp.tanh(z) if i < len(params) - 1 else z
    return a

def fourier_features(xy, n_freq=6, scale=4.0):
    """ویژگی‌های فوریه (Tancik et al., 2020) برای شاخه‌ی اصطکاک -- کمک به
    یادگیری مرز تیز به‌جای فقط الگوهای صاف فرکانس-پایین."""
    freqs = scale * jnp.arange(1, n_freq + 1) * jnp.pi
    ang = xy[..., None] * freqs  # (...,2,n_freq)
    ang = ang.reshape(xy.shape[:-1] + (-1,))
    return jnp.concatenate([xy, jnp.sin(ang), jnp.cos(ang)], axis=-1)

HYDRO_LAYERS = [3, 64, 64, 64, 3]          # (x,y,t)_norm -> (h,u,v)
N_FOURIER = 6
PHI_IN_DIM = 2 + 2 * 2 * N_FOURIER          # (x,y) + sin/cos features
PHI_LAYERS = [PHI_IN_DIM, 32, 32, 1]        # فوریه(x,y) -> phi

def init_params():
    k1 = next_key(); k2 = next_key()
    return {
        "hydro": init_mlp(k1, HYDRO_LAYERS),
        "phi_net": init_mlp(k2, PHI_LAYERS),
        "cf_low": jnp.array(CF_BACKGROUND * 0.7),
        "cf_high": jnp.array(CF_BACKGROUND * 1.3),
    }

def hydro_apply(params, x, y, t):
    xn, yn, tn = normalize_xyt(x, y, t)
    out = mlp_apply(params["hydro"], jnp.stack([xn, yn, tn], axis=-1))
    return out[..., 0], out[..., 1], out[..., 2]   # h, u, v

def phi_apply(params, x, y):
    xn, yn = x / LX, y / LY
    feat = fourier_features(jnp.stack([xn, yn], axis=-1), n_freq=N_FOURIER)
    return mlp_apply(params["phi_net"], feat)[..., 0]

def cf_apply(params, x, y, k):
    """گزینه‌ی الف بخش ۵.۱: سطح-صفر annealed با دو اسکالر یادگرفتنی."""
    phi = phi_apply(params, x, y)
    sig = jax.nn.sigmoid(k * phi)
    return params["cf_low"] + (params["cf_high"] - params["cf_low"]) * sig

def cf_apply_kzone_soft(params_kzone, x, y, temperature):
    """گزینه‌ی ب (فاز ۲، پاسخ به H1d): تخصیص نرم K-ناحیه‌ای.
    params_kzone باید شامل 'zone_net' (خروجی K-بعدی) و 'cf_values' (K اسکالر) باشد.
    اینجا فقط برای تکمیل مشخصات آمده؛ فاز ۰ از cf_apply (گزینه‌ی الف) استفاده می‌کند."""
    logits = mlp_apply(params_kzone["zone_net"], jnp.stack([x / LX, y / LY], axis=-1))
    weights = jax.nn.softmax(logits / temperature, axis=-1)
    return jnp.sum(weights * params_kzone["cf_values"][None, :], axis=-1)

PARAMS0 = init_params()
print("[Cell 5] معماری مقداردهی شد. تعداد پارامتر hydro:",
      sum(w.size + b.size for w, b in PARAMS0["hydro"]),
      " | phi_net:", sum(w.size + b.size for w, b in PARAMS0["phi_net"]))


## Cell 6 — تابع هزینه، Adam دستی، آموزش دو-فازی با گذار Freeze-Release

In [ ]:
# ============================================================
# Cell 6 — تابع هزینه و آموزش Adam (دو فاز + میرایی نرم گشتاورها)
# ============================================================
def pde_residual_batch(params, x, y, t, k, wind_on, diff_on):
    """پسماند معادلات آب کم‌عمق در نقاط کولوکیشن، با autodiff دقیق."""
    def single(x_, y_, t_):
        h_fn = lambda xx, yy, tt: hydro_apply(params, xx, yy, tt)[0]
        u_fn = lambda xx, yy, tt: hydro_apply(params, xx, yy, tt)[1]
        v_fn = lambda xx, yy, tt: hydro_apply(params, xx, yy, tt)[2]

        h_t = jax.grad(h_fn, argnums=2)(x_, y_, t_)
        u_t = jax.grad(u_fn, argnums=2)(x_, y_, t_)
        v_t = jax.grad(v_fn, argnums=2)(x_, y_, t_)
        h_x = jax.grad(h_fn, argnums=0)(x_, y_, t_)
        h_y = jax.grad(h_fn, argnums=1)(x_, y_, t_)
        u_x = jax.grad(u_fn, argnums=0)(x_, y_, t_)
        v_y = jax.grad(v_fn, argnums=1)(x_, y_, t_)

        _, u_, v_ = hydro_apply(params, x_, y_, t_)
        cf_ = cf_apply(params, x_[None], y_[None], k)[0]
        speed = jnp.sqrt(u_**2 + v_**2 + 1e-8)

        wind_x = jnp.where(wind_on, WIND_AMPLITUDE * jnp.sin(2 * jnp.pi * y_ / LY), 0.0)

        if diff_on:
            u_xx = jax.grad(lambda xx: jax.grad(u_fn, argnums=0)(xx, y_, t_))(x_)
            u_yy = jax.grad(lambda yy: jax.grad(u_fn, argnums=1)(x_, yy, t_))(y_)
            diff_u = NU_DIFF * (u_xx + u_yy)
        else:
            diff_u = 0.0

        res_u = u_t + G * h_x - F0 * v_ + (cf_ / H0) * u_ * speed - wind_x - diff_u
        res_v = v_t + G * h_y + F0 * u_ + (cf_ / H0) * v_ * speed
        res_continuity = h_t + H0 * (u_x + v_y)
        return res_u, res_v, res_continuity

    ru, rv, rc = jax.vmap(single)(x, y, t)
    return ru, rv, rc

def velocity_cap_penalty(params):
    _, u_c, v_c = jax.vmap(lambda x, y, t: hydro_apply(params, x, y, t))(COLL_X, COLL_Y, COLL_T)
    over_u = jax.nn.relu(jnp.abs(u_c) - VELOCITY_REG_CAP_MPS)
    over_v = jax.nn.relu(jnp.abs(v_c) - VELOCITY_REG_CAP_MPS)
    return jnp.mean(over_u**2) + jnp.mean(over_v**2)

def zone_separation_penalty(params):
    gap = jnp.abs(params["cf_high"] - params["cf_low"])
    return jax.nn.relu(ZONE_SEPARATION_MARGIN - gap) ** 2

# --- نکته‌ی حیاتی JAX: چرا loss_and_grad یک "کارخانه" (factory) است ---
# اگر total_loss مستقیماً روی متغیرهای سراسری (WIND_ON, H_OBS, ...) بسته شود و فقط
# یک‌بار jax.jit شود، تغییر این متغیرها بین سناریوهای Cell 9 اثری روی نسخه‌ی
# کامپایل‌شده نخواهد داشت (jax.jit مقادیر بسته‌شده را در زمان اولین trace ثابت
# می‌کند). راه‌حل: هر بار قبل از آموزش یک تابع total_loss تازه (با مقادیر
# جاری wind_on/diff_on/observe_velocity/h_obs/u_obs/v_obs) ساخته و jit می‌شود.
def build_loss_and_grad(wind_on, diff_on, observe_velocity, h_obs, u_obs, v_obs):
    n_snap = h_obs.shape[0]
    flat_x = jnp.tile(OBS_X, (n_snap,))
    flat_y = jnp.tile(OBS_Y, (n_snap,))
    flat_t = jnp.repeat(OBS_T, N_OBS_PTS)

    def data_loss(params, k):
        h_pred = jax.vmap(lambda x, y, t: hydro_apply(params, x, y, t)[0])(flat_x, flat_y, flat_t)
        h_pred = h_pred.reshape(n_snap, N_OBS_PTS)
        loss = jnp.mean((h_pred - h_obs) ** 2)
        if observe_velocity:
            u_pred = jax.vmap(lambda x, y, t: hydro_apply(params, x, y, t)[1])(
                flat_x, flat_y, flat_t).reshape(n_snap, N_OBS_PTS)
            v_pred = jax.vmap(lambda x, y, t: hydro_apply(params, x, y, t)[2])(
                flat_x, flat_y, flat_t).reshape(n_snap, N_OBS_PTS)
            loss = loss + jnp.mean((u_pred - u_obs) ** 2) + jnp.mean((v_pred - v_obs) ** 2)
        return loss

    def physics_loss(params, k):
        ru, rv, rc = pde_residual_batch(params, COLL_X, COLL_Y, COLL_T, k, wind_on, diff_on)
        return jnp.mean(ru**2) + jnp.mean(rv**2) + jnp.mean(rc**2)

    def total_loss(params, k, phys_weight):
        return (data_loss(params, k)
                + phys_weight * physics_loss(params, k)
                + 1.0 * velocity_cap_penalty(params)
                + ZONE_SEPARATION_WEIGHT * zone_separation_penalty(params))

    return jax.jit(jax.value_and_grad(total_loss))

# ---------------- Adam دستی (پیچ‌های pytree keras-free) ----------------
def adam_init(params):
    zeros = jax.tree_util.tree_map(jnp.zeros_like, params)
    return {"m": zeros, "v": jax.tree_util.tree_map(jnp.zeros_like, params), "t": 0}

@jax.jit
def adam_step(params, grads, state, lr):
    t = state["t"] + 1
    b1, b2, eps = 0.9, 0.999, 1e-8
    m = jax.tree_util.tree_map(lambda m_, g: b1 * m_ + (1 - b1) * g, state["m"], grads)
    v = jax.tree_util.tree_map(lambda v_, g: b2 * v_ + (1 - b2) * g**2, state["v"], grads)
    m_hat = jax.tree_util.tree_map(lambda m_: m_ / (1 - b1**t), m)
    v_hat = jax.tree_util.tree_map(lambda v_: v_ / (1 - b2**t), v)
    new_params = jax.tree_util.tree_map(
        lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps), params, m_hat, v_hat)
    return new_params, {"m": m, "v": v, "t": t}

def decay_subtree_moments(state, subtree_key, decay):
    """میرایی نرم گشتاورهای Adam فقط برای یک زیردرخت (اینجا: phi_net) --
    جایگزین بازنشانی آنی که باعث جهش loss در epoch گذار می‌شد (بخش ۲.۳/۵.۳ گزارش)."""
    m = dict(state["m"]); v = dict(state["v"])
    m[subtree_key] = jax.tree_util.tree_map(lambda a: a * decay, m[subtree_key])
    v[subtree_key] = jax.tree_util.tree_map(lambda a: a * decay, v[subtree_key])
    return {"m": m, "v": v, "t": state["t"]}

def train(params, wind_on, diff_on, observe_velocity, h_obs, u_obs, v_obs, lr=1e-3, log_every=None):
    """یک اجرای آموزش کامل برای یک سناریو. هر بار یک loss_and_grad تازه
    می‌سازد (به دلیل نکته‌ی بالا درباره‌ی jit + closure) و آن را برمی‌گرداند
    تا Cell 7 (L-BFGS) هم دقیقاً همان تابع هزینه را ادامه دهد."""
    log_every = log_every or max(1, (ADAM_EPOCHS_PHASE1 + ADAM_EPOCHS_PHASE2) // 10)
    loss_and_grad = build_loss_and_grad(wind_on, diff_on, observe_velocity, h_obs, u_obs, v_obs)
    state = adam_init(params)
    history = []

    print("[Cell 6] فاز ۱ (وزن فیزیک پایین) ...")
    for epoch in range(ADAM_EPOCHS_PHASE1):
        loss, grads = loss_and_grad(params, K_MIN, PHYS_WEIGHT_PHASE1)
        params, state = adam_step(params, grads, state, lr)
        history.append(float(loss))
        if epoch % log_every == 0:
            print(f"  فاز۱ epoch {epoch:5d}  loss={loss:.6e}")

    print("[Cell 6] گذار Freeze-Release: میرایی نرم گشتاورهای phi_net ...")
    print("[Cell 6] فاز ۲ (آزادسازی فیزیک + annealing k) ...")
    for epoch in range(ADAM_EPOCHS_PHASE2):
        if epoch < WARM_TRANSITION_EPOCHS:
            state = decay_subtree_moments(state, "phi_net", MOMENT_DECAY_RATE)
        k = k_schedule(epoch)
        loss, grads = loss_and_grad(params, k, PHYS_WEIGHT_PHASE2)
        params, state = adam_step(params, grads, state, lr)
        history.append(float(loss))
        if epoch % log_every == 0:
            print(f"  فاز۲ epoch {epoch:5d}  loss={loss:.6e}  k={float(k):.2f}")

    return params, history, loss_and_grad

print("[Cell 6] آماده -- train(...) در Cell 9 (ablation) برای هر سناریو صدا زده می‌شود.")


## Cell 7 — پرداخت نهایی با L-BFGS (full-batch، قطعی)

In [ ]:
# ============================================================
# Cell 7 — L-BFGS (پس از Adam، طبق بخش ۵.۴ گزارش)
# ============================================================
from scipy.optimize import minimize

def flatten_params(params):
    leaves, treedef = jax.tree_util.tree_flatten(params)
    shapes = [l.shape for l in leaves]
    flat = np.concatenate([np.asarray(l).ravel() for l in leaves])
    return flat, treedef, shapes

def unflatten_params(flat, treedef, shapes):
    leaves = []
    idx = 0
    for shp in shapes:
        n = int(np.prod(shp)) if shp else 1
        leaves.append(jnp.array(flat[idx:idx + n]).reshape(shp))
        idx += n
    return jax.tree_util.tree_unflatten(treedef, leaves)

def lbfgs_finetune(params, loss_and_grad, k_final, maxiter=None):
    """loss_and_grad باید همان تابعی باشد که train() برای این سناریو برگرداند
    (همان wind_on/diff_on/observe_velocity/h_obs/u_obs/v_obs بسته‌شده در آن)."""
    maxiter = maxiter or LBFGS_MAXITER
    flat0, treedef, shapes = flatten_params(params)

    def loss_fn_np(flat):
        p = unflatten_params(flat, treedef, shapes)
        loss, grads = loss_and_grad(p, k_final, PHYS_WEIGHT_PHASE2)
        gflat, _, _ = flatten_params(grads)
        return float(loss), np.asarray(gflat, dtype=np.float64)

    print(f"[Cell 7] شروع L-BFGS (full-batch، قطعی)، حداکثر {maxiter} تکرار ...")
    res = minimize(loss_fn_np, flat0, jac=True, method="L-BFGS-B",
                    options={"maxiter": maxiter, "disp": False})
    print(f"[Cell 7] L-BFGS پایان یافت: loss نهایی={res.fun:.6e}, تعداد تکرار={res.nit}")
    return unflatten_params(res.x, treedef, shapes)


## Cell 8 — ارزیابی (RMSE، r(Cf)، AUC ناحیه، خطای مرز، کنترل باد)

In [ ]:
# ============================================================
# Cell 8 — ارزیابی
# ============================================================
def predicted_cf_field(params, k_final):
    xc = (jnp.arange(NX) + 0.5) * DX
    yc = (jnp.arange(NY) + 0.5) * DY
    Xc, Yc = jnp.meshgrid(xc, yc, indexing="ij")
    return cf_apply(params, Xc.ravel(), Yc.ravel(), k_final).reshape(NX, NY)

def rmse_r_metrics(cf_pred, cf_true):
    cf_pred = np.asarray(cf_pred); cf_true = np.asarray(cf_true)
    rng = cf_true.max() - cf_true.min()
    rmse_pct = 100 * np.sqrt(np.mean((cf_pred - cf_true) ** 2)) / rng
    r = np.corrcoef(cf_pred.ravel(), cf_true.ravel())[0, 1]
    return rmse_pct, r

def two_zone_scores(cf_pred, cf_true):
    """AUC طبقه‌بندی ناحیه (بخش ۶ گزارش) + خطای موقعیت مرز (فاصله‌ی میانگین سلولی)."""
    from numpy import corrcoef
    cf_pred = np.asarray(cf_pred); cf_true = np.asarray(cf_true)
    true_label = (cf_true > 0.5 * (CF_LOW_TRUE + CF_HIGH_TRUE)).astype(int).ravel()
    score = cf_pred.ravel()  # مقدار پیوسته به‌عنوان امتیاز طبقه‌بندی

    # AUC دستی (بدون sklearn) از طریق شمارش جفت‌های صحیح‌رتبه‌بندی‌شده
    pos = score[true_label == 1]; neg = score[true_label == 0]
    if len(pos) == 0 or len(neg) == 0:
        auc = float("nan")
    else:
        auc = np.mean(pos[:, None] > neg[None, :]) + 0.5 * np.mean(pos[:, None] == neg[None, :])

    pred_label = (cf_pred > np.median(cf_pred)).astype(int)
    # خطای مرز: فاصله‌ی میانگین بین سلول‌های مرزی پیش‌بینی‌شده و واقعی (به تعداد سلول شبکه)
    def boundary_cells(label_grid):
        edges = np.zeros_like(label_grid, dtype=bool)
        edges[:-1, :] |= label_grid[:-1, :] != label_grid[1:, :]
        edges[:, :-1] |= label_grid[:, :-1] != label_grid[:, 1:]
        return np.argwhere(edges)
    true_edges = boundary_cells((cf_true > 0.5 * (CF_LOW_TRUE + CF_HIGH_TRUE)).astype(int))
    pred_edges = boundary_cells(pred_label)
    if len(true_edges) == 0 or len(pred_edges) == 0:
        boundary_err = float("nan")
    else:
        d = np.sqrt(((pred_edges[:, None, :] - true_edges[None, :, :]) ** 2).sum(-1))
        boundary_err = d.min(axis=1).mean()
    return auc, boundary_err

def wind_correlation_controls(cf_pred, cf_true):
    """کنترل صریح: هم‌بستگی خطای Cf با الگوی باد، در برابر کنترل حقیقی/تصادفی (بخش ۲.۴ گزارش)."""
    wind_2d = np.asarray(jnp.tile(WIND_PATTERN_U[0][None, :], (NX, 1)))
    err = np.asarray(cf_pred) - np.asarray(cf_true)
    corr_err = np.corrcoef(err.ravel(), wind_2d.ravel())[0, 1]
    corr_true = np.corrcoef(np.asarray(cf_true).ravel(), wind_2d.ravel())[0, 1]
    corr_random = np.corrcoef(_rng4.normal(size=wind_2d.size), wind_2d.ravel())[0, 1]
    return corr_err, corr_true, corr_random

def evaluate(params, k_final, label=""):
    cf_pred = predicted_cf_field(params, k_final)
    rmse_pct, r = rmse_r_metrics(cf_pred, CF_TRUE)
    auc, boundary_err = two_zone_scores(cf_pred, CF_TRUE)
    corr_err, corr_true, corr_random = wind_correlation_controls(cf_pred, CF_TRUE)
    result = dict(label=label, patch_level=PATCH_LEVEL, run_mode=RUN_MODE,
                  wind_on=WIND_ON, diff_on=DIFF_ON, observe_velocity=OBSERVE_VELOCITY,
                  rmse_pct=rmse_pct, r=r, auc=auc, boundary_err_cells=boundary_err,
                  corr_err_wind=corr_err, corr_true_wind=corr_true, corr_random_wind=corr_random,
                  cf_low=float(params["cf_low"]), cf_high=float(params["cf_high"]))
    print(f"[Cell 8] {label}: RMSE={rmse_pct:.1f}%  r={r:.3f}  AUC={auc:.3f}  "
          f"boundary_err={boundary_err:.2f} cells")
    return result, cf_pred


## Cell 9 — ارکستراسیون Ablation (اجرای همه‌ی سناریوها + ذخیره‌ی نتایج)

In [ ]:
# ============================================================
# Cell 9 — Ablation: اجرای سناریوها و ذخیره‌ی جدول نتایج
# ============================================================
def run_scenario(wind_on, diff_on, observe_velocity, label):
    global WIND_ON, DIFF_ON, OBSERVE_VELOCITY, ETA_TRUE, U_TRUE, V_TRUE
    global _h_clean, _u_clean, _v_clean, H_OBS, U_OBS, V_OBS

    WIND_ON, DIFF_ON, OBSERVE_VELOCITY = wind_on, diff_on, observe_velocity

    # داده‌ی مرجع باید با پرچم‌های wind/diff فعلی دوباره تولید شود
    ETA_TRUE, U_TRUE, V_TRUE = run_reference_solver(CF_TRUE, WIND_ON, DIFF_ON)
    _h_clean = _sample_traj(ETA_TRUE, _ix_obs, _iy_obs, _snapshot_steps)
    _u_clean = _sample_traj(U_TRUE, _ix_obs, _iy_obs, _snapshot_steps)
    _v_clean = _sample_traj(V_TRUE, _ix_obs, _iy_obs, _snapshot_steps)
    H_OBS = jnp.array(_h_clean + _rng4.normal(scale=_noise_std, size=_h_clean.shape))
    U_OBS = jnp.array(_u_clean + _rng4.normal(scale=_noise_std, size=_u_clean.shape))
    V_OBS = jnp.array(_v_clean + _rng4.normal(scale=_noise_std, size=_v_clean.shape))

    params = init_params()
    params, history, loss_and_grad = train(params, WIND_ON, DIFF_ON, OBSERVE_VELOCITY, H_OBS, U_OBS, V_OBS)
    params = lbfgs_finetune(params, loss_and_grad, k_final=K_MAX)
    result, cf_pred = evaluate(params, K_MAX, label=label)
    return result, params, cf_pred, history

SCENARIOS = [
    dict(wind_on=False, diff_on=False, observe_velocity=False, label="Wind=OFF, Diff=OFF (h-تنها)"),
    dict(wind_on=True,  diff_on=False, observe_velocity=False, label="Wind=ON, Diff=OFF (h-تنها)"),
    dict(wind_on=True,  diff_on=True,  observe_velocity=False, label="Wind=ON, Diff=ON (h-تنها)"),
    dict(wind_on=True,  diff_on=True,  observe_velocity=True,  label="Wind=ON, Diff=ON + u,v OBSERVED (کنترل)"),
]

if __name__ == "__main__":
    all_results = []
    for sc in SCENARIOS:
        print(f"\n{'='*60}\n[Cell 9] سناریو: {sc['label']}\n{'='*60}")
        result, params, cf_pred, history = run_scenario(**sc)
        all_results.append(result)

    with open(f"ablation_results_{RUN_MODE}.pkl", "wb") as f:
        pickle.dump(all_results, f)

    print("\n\n========== جدول خلاصه (مقایسه با جدول بخش ۴.۲ گزارش) ==========")
    print(f"{'سناریو':<45} {'RMSE%':>8} {'r':>8} {'AUC':>8} {'boundary_err':>13}")
    for r in all_results:
        print(f"{r['label']:<45} {r['rmse_pct']:>8.1f} {r['r']:>8.3f} {r['auc']:>8.3f} {r['boundary_err_cells']:>13.2f}")
    print(f"\nنتایج در ablation_results_{RUN_MODE}.pkl ذخیره شد.")
